# Appendix Table XXV — Embedding Sensitivity

Reproduce the topic-encoder embedding sensitivity experiment reported in **Appendix Table XXV**.

- 7 topic encoders
- 5 random seeds
- Mixture sizes $K \in \{2,3,4,5,6\}$
- Metric: **Macro-F1 (mean ± sample standard deviation)**
- Evaluation: **fixed-slot / without Hungarian remapping**


## 1. Environment

Recommended environment:

| Component | Version |
|---|---:|
| Python | 3.12 |
| PyTorch | 2.6 |
| NumPy | 2.1 |
| pandas | 2.2 |
| SciPy | 1.15 |
| scikit-learn | 1.6 |

CUDA 11.8+ is recommended for the full run. CPU execution is supported but slower.


## 2. Required evaluation assets

Place the following files **in the same directory as this notebook**:

1. `synthetic_validation.tar.xz`
2. `checkpoints_tgcm.tar.xz.part-000`
3. `checkpoints_tgcm.tar.xz.part-001`

`synthetic_validation.tar.xz` contains the evaluation-only validation sequences for $K=2,...,6$. The two checkpoint parts contain the sanitized TGCM inference checkpoints for all seven embedding families and five seeds.

**Not required:** training data, optimizer state, API keys, CAPTure raw CSV files, or outputs from any other notebook.


In [ ]:
from pathlib import Path
import shutil
import sys
import urllib.request

SUPPORT_REV = "de9a81f726ebff439539edf345be98f619844fc8"
RAW_BASE = f"https://raw.githubusercontent.com/Irish-kw/TGCM_Website/{SUPPORT_REV}/reproduction"
NOTEBOOK_DIR = Path.cwd().resolve()

def find_repository_root():
    for candidate in (NOTEBOOK_DIR, *NOTEBOOK_DIR.parents):
        if (candidate / "tgcm_review").is_dir() and (candidate / "data").is_dir():
            return candidate
    return None

ROOT = find_repository_root()

if ROOT is None:
    ROOT = NOTEBOOK_DIR / "tgcm_reproduction_support"
    support_files = [
        "tgcm_review/__init__.py",
        "tgcm_review/assets.py",
        "tgcm_review/datasets.py",
        "tgcm_review/metrics.py",
        "tgcm_review/models.py",
        "tgcm_review/inference.py",
        "tgcm_review/paper_experiments.py",
        "data/manifest.json",
        "environment.yml",
    ]
    for relative_path in support_files:
        destination = ROOT / relative_path
        destination.parent.mkdir(parents=True, exist_ok=True)
        if not destination.exists():
            print(f"Downloading support file: {relative_path}")
            urllib.request.urlretrieve(f"{RAW_BASE}/{relative_path}", destination)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Support root: {ROOT}")


In [ ]:
ASSET_DIR = ROOT / "data" / "assets"
ASSET_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_ASSETS = [
    "synthetic_validation.tar.xz",
    "checkpoints_tgcm.tar.xz.part-000",
    "checkpoints_tgcm.tar.xz.part-001",
]

# If the notebook was downloaded by itself, copy assets placed beside it
# into the directory expected by the reproduction helper.
for filename in REQUIRED_ASSETS:
    local_file = NOTEBOOK_DIR / filename
    target_file = ASSET_DIR / filename
    if local_file.is_file() and not target_file.exists():
        shutil.copy2(local_file, target_file)

missing = [name for name in REQUIRED_ASSETS if not (ASSET_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(
        "Missing required evaluation assets:\n  - "
        + "\n  - ".join(missing)
        + f"\n\nPlace them beside this notebook or under:\n{ASSET_DIR}"
    )

print("All required evaluation assets are present.")


## 3. Run the embedding-sensitivity experiment

The full paper reproduction evaluates every encoder for all five seeds and all five mixture sizes. Set `FULL_REPRODUCTION = False` only for a quick code-path smoke test; smoke-test values are **not** paper results.


In [ ]:
from tgcm_review.paper_experiments import run_figure04

FULL_REPRODUCTION = True
DEVICE = None  # None selects CUDA when available; use "cpu" explicitly for CPU.
MAX_SEQUENCES = None if FULL_REPRODUCTION else 512

detail, summary = run_figure04(
    components=("tgcm",),
    device=DEVICE,
    max_sequences=MAX_SEQUENCES,
    root=ROOT,
)

print("Full paper run completed." if FULL_REPRODUCTION else "Smoke test completed — not a paper result.")


## 4. Format Appendix Table XXV

Select the paper's **fixed-slot** evaluation (`without_hungarian`) and report Macro-F1 as mean ± sample standard deviation across the five seeds.


In [ ]:
import pandas as pd

table25 = summary[
    summary["alignment_scope"].eq("without_hungarian")
    & summary["metric"].eq("Macro-F1")
].copy()

table25["Embedding"] = table25["model"].str.extract(r"TGCM \((.*)\)", expand=False)
table25 = table25[["K", "Embedding", "mean", "std"]].sort_values(["K", "Embedding"])

paper_table = (
    table25.assign(
        value=table25.apply(lambda row: f"{row['mean']:.3f} ± {row['std']:.3f}", axis=1)
    )
    .pivot(index="K", columns="Embedding", values="value")
    .reindex(columns=[
        "ATTACK-BERT",
        "CTI-BERT",
        "CYBERT",
        "CySecBERT",
        "SecBERT",
        "SecureBERT",
        "all-MiniLM-L6-v2",
    ])
)

paper_table
